In [ ]:
# --- setup / imports (deduped) ---
import os 
os.chdir(r'/Users/sachuriga/Desktop/code/nwb4fp/SRC')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import math
import pynapple as nap
from scipy import signal
from sklearn.preprocessing import normalize
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from scipy.stats import gaussian_kde
from scipy import stats # ADD THIS IMPORT
import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
from itertools import chain
from nwb4fp.data.helpers import df2results, df2results_sns


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
import statsmodels.formula.api as smf
import warnings as _w


def _locomotion_lmm_p(control_df, exp_df, metric):
    """Session-level p: metric ~ genotype + (1|animal). Returns (p, n_sessions, n_mice)."""
    a = control_df[['animal_id', metric]].copy(); a['g'] = 'control'
    b = exp_df[['animal_id', metric]].copy();     b['g'] = 'exp'
    d = pd.concat([a, b], ignore_index=True)
    d['y'] = pd.to_numeric(d[metric], errors='coerce')
    d = d.dropna(subset=['y'])
    d['animal_id'] = d['animal_id'].astype(str)
    d['g'] = pd.Categorical(d['g'], categories=['control', 'exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            m = smf.mixedlm("y ~ g", d, groups=d['animal_id']).fit(reml=True)
            return m.pvalues.get('g[T.exp]', np.nan), len(d), d['animal_id'].nunique()
        except Exception:
            return np.nan, len(d), d['animal_id'].nunique()

# ==========================================
# 1. Configuration & Constants
# ==========================================
# File Paths
BASE_FOLDER = r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Results/Results/Locomotion"
DATA_PATH = '/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/speed_analysis_results.pkl'
SAVE_PATH = r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw/suppfig3.pdf'

# Animal IDs
CONTROL_IDS = ['65165', '65091', '63383', '66539', '65622']
EXP_IDS = ['65588', '63385', '66538', '66537', '66922']

# Plotting Configuration
plt.rcParams.update({
    'font.size': 7,
    'pdf.fonttype': 42,   # keep text editable in Illustrator
    'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Calibri', 'DejaVu Sans', 'sans-serif'],
    'axes.labelpad': 5,
    'ytick.major.pad': 2,
    'xtick.major.pad': 5,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

# ==========================================
# 2. Helper Functions
# ==========================================

# turn the x/y positions into a normalized 2D occupancy map (KDE)
def compute_normalized_kde(data, x="x", y="y", gridsize=100):
    """Computes normalized 2D KDE density."""
    xy = np.vstack([data[x], data[y]])
    kde = gaussian_kde(xy)
    
    x_grid = np.linspace(0, 1, gridsize)
    y_grid = np.linspace(0, 1, gridsize)
    X, Y = np.meshgrid(x_grid, y_grid)
    positions = np.vstack([X.ravel(), Y.ravel()])
    
    density = kde(positions).reshape(gridsize, gridsize)
    # Normalize to [0, 1]
    density = (density - density.min()) / (density.max() - density.min())
    return X, Y, density

# keep the heatmap axes square and drop the spines
def format_square_axis(ax):
    """Applies common formatting for the top row square plots."""
    ax.set_xlabel('cm')
    ax.set_ylabel('cm')
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_visible(False)

# ==========================================
# 3. Data Loading & Preparation
# ==========================================

# Load Data
# load the speed/locomotion results
df = pd.read_pickle(DATA_PATH)

# Filter for Session A
df_a = df[df['session'] == "A"]

# Split into Control and Experimental
control_df = df_a[df_a['animal_id'].isin(CONTROL_IDS)]
exp_df = df_a[df_a['animal_id'].isin(EXP_IDS)]

# NOTE: The following functions (df2results_sns, df2results) are assumed 
# to be defined in your environment or imported from another module.
# Ensure these are available before running.

# ==========================================
# 4. Figure Layout Setup
# ==========================================

fig = plt.figure(figsize=(7.2, 4.1), dpi=1200)

# Main Grid: 2 rows (heights equal) -> Top part vs Bottom part
gs_main = gridspec.GridSpec(2, 4, height_ratios=[1, 1], hspace=0.55, wspace=0.45)

# Top Row Subplots (1-4)
ax1 = fig.add_subplot(gs_main[0, 0]) # Control KDE
ax2 = fig.add_subplot(gs_main[0, 1]) # Exp Heatmap
ax3 = fig.add_subplot(gs_main[0, 2]) # Exp KDE
ax4 = fig.add_subplot(gs_main[0, 3]) # Control Heatmap

# Bottom Row Subplots (5-10) using a nested GridSpec
gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 6, subplot_spec=gs_main[1, :], wspace=0.95)
bottom_axes = [fig.add_subplot(gs_bottom[i]) for i in range(6)]

# ==========================================
# 5. Plotting: Top Row (KDEs & Heatmaps)
# ==========================================

# --- Plot 1: Control KDE (Top-Left) ---
X, Y, density_control = compute_normalized_kde(df2results_sns(control_df))
cf1 = ax1.contourf(X, Y, density_control, levels=100, cmap="Blues", vmin=0, vmax=1)
cbar1 = fig.colorbar(cf1, ax=ax1, shrink=0.5, aspect=10, pad=0.02, ticks=[0, 0.5, 1])
cbar1.outline.set_visible(False)
cbar1.set_label("Duration")

ax1.set_xticks([0, 1])
ax1.set_xticklabels([0, 50])
ax1.set_yticks([0, 1])
ax1.set_yticklabels([0, 50])
format_square_axis(ax1)

# --- Plot 2: Experimental Heatmap (Top-Right 1) ---
data_speedss_exp = df2results(exp_df)
n_rows, n_cols = data_speedss_exp.shape

sns.heatmap(data_speedss_exp, cmap="Blues", annot=False, linewidths=0, vmin=0, ax=ax2, square=True,
            xticklabels=np.linspace(0, 50, 6), yticklabels=np.linspace(50, 0, 6),
            cbar_kws={'shrink': 0.5, 'aspect': 10, 'pad': 0.02})
ax2.collections[0].colorbar.set_label("Speed (cm/s)")

ax2.set_xticks([0, n_cols - 1])
ax2.set_xticklabels([0, 50])
ax2.set_yticks([0, n_rows - 1])
ax2.set_yticklabels([50, 0])
format_square_axis(ax2)

# --- Plot 3: Experimental KDE (Top-Right 2) ---
X_exp, Y_exp, density_exp = compute_normalized_kde(df2results_sns(exp_df))
cf3 = ax3.contourf(X_exp, Y_exp, density_exp, levels=100, cmap="Reds", vmin=0, vmax=1)
cbar3 = fig.colorbar(cf3, ax=ax3, shrink=0.5, aspect=10, pad=0.02, ticks=[0, 0.5, 1])
cbar3.outline.set_visible(False)
cbar3.set_label("Duration")

ax3.set_xticks([0, 1])
ax3.set_xticklabels([0, 50])
ax3.set_yticks([0, 1])
ax3.set_yticklabels([0, 50])
format_square_axis(ax3)

# --- Plot 4: Control Heatmap (Top-Right 3) ---
data_speedss_control = df2results(control_df)
n_rows_c, n_cols_c = data_speedss_control.shape

sns.heatmap(data_speedss_control, cmap="Reds", annot=False, linewidths=0, vmin=0, ax=ax4, square=True,
            xticklabels=np.linspace(0, 50, 6), yticklabels=np.linspace(50, 0, 6),
            cbar_kws={'shrink': 0.5, 'aspect': 10, 'pad': 0.02})
ax4.collections[0].colorbar.set_label("Speed (cm/s)")

ax4.set_xticks([0, n_cols_c - 1])
ax4.set_xticklabels([0, 50])
ax4.set_yticks([0, n_rows_c - 1])
ax4.set_yticklabels([50, 0])
format_square_axis(ax4)

# ==========================================
# 6. Plotting: Bottom Row (Stats Boxplots & Legend Generation)
# ==========================================

titles = ['Speed in center (cm/s)', 'Speed in edge (cm/s)', 'Center-edge\nspeed ratio',
          'Active times (s)', "Average speed (cm/s)", "Time in center (s)"]

metrics = ['filter_speed_in_center', 'filter_speed_in_edge', 'center_border_ratio', 
           'active_times', 'mean_speed', 'time_in_center']

# Pre-calculate means per animal
control_animals = control_df.groupby('animal_id').mean(numeric_only=True)
exp_animals = exp_df.groupby('animal_id').mean(numeric_only=True)

# List to store the generated p-values for the printed legend
legend_stats_text = []
letters = ['E', 'F', 'G', 'H', 'I', 'J']

# go metric by metric: run the test, plot it, slap the p-value on top
for idx, metric in enumerate(metrics):
    ax = bottom_axes[idx]
    
    # Extract data
    control_values = control_animals[metric].dropna()
    exp_values = exp_animals[metric].dropna()
    
    if len(control_values) > 0 and len(exp_values) > 0:
        
        # --- 1. RUN STATISTICAL TEST ---
        # Animal-level: linear mixed model on session-level values with animal as a
        # random intercept (not a t-test on 5 collapsed animal means), so the locomotion
        # panels use the same statistical framework as the rest of the revision.
        p_val, n_sess, n_mice = _locomotion_lmm_p(control_df, exp_df, metric)
        n_ctrl = len(control_values)   # animals per group, for the SuperPlot points
        n_exp = len(exp_values)
        legend_stats_text.append(f"({letters[idx]}) LMM p = {p_val:.4f} "
                                 f"(n = {n_sess} sessions from {n_mice} mice)")
        
        # --- 2. PREPARE DATA FOR PLOTTING ---
        plot_df = pd.DataFrame({
            'value': pd.concat([control_values, exp_values]),
            'group': ['Control'] * n_ctrl + ['Experimental'] * n_exp
        })
        
        # Boxplot
        sns.boxplot(
            data=plot_df, x='group', y='value', ax=ax,
            palette={"Control": (0, 0, 1), "Experimental": (1, 0, 0)},
            whis=[0, 100], width=.6,
            boxprops=dict(edgecolor=None),
            whiskerprops=dict(color="black")
        )
        
        # Stripplot
        sns.stripplot(
            data=plot_df, x='group', y='value', ax=ax,
            size=2, hue='group',
            palette={"Control": "#ECE0CA", "Experimental": "#ECE0CA"},
            alpha=1, jitter=0.1, legend=False
        )
        
        # --- 3. ADD P-VALUE TO THE GRAPH ---
        y_max = plot_df['value'].max()
        y_min = plot_df['value'].min()
        y_range = y_max - y_min
        
        # Draw a line between the two groups
        line_y = y_max + 0.05 * y_range
        ax.plot([0, 1], [line_y, line_y], lw=1, c='black')
        
        # Determine asterisks or exact p-value text
        if p_val < 0.0001:
            sig_text = "****"
        elif p_val < 0.001:
            sig_text = "***"
        elif p_val < 0.01:
            sig_text = "**"
        elif p_val < 0.05:
            sig_text = "*"
        else:
            sig_text = "ns"
            
        # Add the text above the line (you can change sig_text to f"p={p_val:.3f}" if you prefer exact numbers on the plot)
        ax.text(0.5, line_y + 0.02 * y_range, sig_text, ha='center', va='bottom', color='black', fontsize=8)
        
        # Adjust y-axis limits to fit the new text and line
        ax.set_ylim(bottom=0, top=y_max + 0.2 * y_range)

        # Formatting
        ax.set_ylabel(titles[idx])
        ax.set_xlabel(metric)
        ax.xaxis.label.set_visible(False)
        ax.yaxis.grid(False)
        ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-25)
        
        # Spine formatting
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)

# ==========================================
# 7. Final Output & Legend Generation
# ==========================================

# only the leftmost heatmap keeps the 'cm' y-label; the others are redundant and
# collided with the neighbouring colorbar labels
for _a in [ax2, ax3, ax4]:
    _a.set_ylabel('')

plt.savefig(SAVE_PATH, transparent=True, dpi=1200, bbox_inches='tight')

# Print the auto-generated figure legend text to the console
print("\n" + "="*60)
print("AUTO-GENERATED STATISTICAL LEGEND FOR NEURON:")
print("="*60)
print(f"Data are represented as median with interquartile range (IQR), and whiskers showing minimum and maximum values. "
      f"Individual data points represent individual mice (Control n={n_ctrl}, Experimental n={n_exp}). "
      f"Statistical significance was determined with linear mixed-effects models (metric ~ genotype, animal as random intercept). "
      f"Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001, **** p < 0.0001, ns = not significant. "
      f"Exact p-values: {'; '.join(legend_stats_text)}.")
print("="*60 + "\n")

plt.show()